# Adaptive Agentic Retrieval Controller for KB-VQA (Zero-Leakage)
### Phase 2: Evidence-Driven Failure Recovery on InfoSeek (1,000 Test Queries)
**Framework Base:** OMGM (*ACL 2025 Main*, arXiv:2505.07879)
**Project Master Plan:** `others/plan and guidence/plan_init.md` (Phase 2 & 3)

---
### Key Architecture & Zero-Leakage Principles:
1. **Zero Test-Time Ground-Truth Leakage:** The decision controller operates in a strictly unsupervised manner. Ground truth answers (`ground_truth_answers`) are never accessed or matched inside the decision loop.
2. **True Symmetric De-biased Visual Choice:** Evaluates both `(A: title1, B: title2)` and `(B: title2, A: title1)` permutations. Overcomes VLM Option-(A) positional bias by requiring unanimous or bias-breaking votes for Candidate 2; positional ties default to Candidate 1 (the higher-ranked retriever top-1).
3. **Dual Category B & C Section Selection:** Uses BGE Section Reranker (`BAAI/bge-reranker-v2-m3` in FP16) to find the most relevant evidence section for Candidate 1 (fixing Category C errors) and Candidate 2 (fixing Category B errors).
4. **Exact InfoSeek Standard Metric:** Strictly adheres to OMGM official benchmark evaluation (+/- 10% numeric tolerance and alias substring matching).

## Cell 1: Environment & GPU Hardware Sanity Check
Verifies CUDA availability, device name, and free VRAM.

In [ ]:
# 0. Dependencies for 4-bit LLaVA and BGE Reranker
!pip install -q -U bitsandbytes accelerate

import os
import sys
import gc
import json
import re
import string
import time
import random
import numpy as np
import pandas as pd
import PIL.Image
from tqdm import tqdm
import torch

# 1. Deterministic Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed(42)

# 2. Hardware Sanity Check
print("=" * 60)
print("SANITY CHECK 1: GPU & ENVIRONMENT")
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device Name : {gpu_name}")
    print(f"Total GPU VRAM  : {vram_gb:.2f} GB")
else:
    print("[WARNING] CUDA not detected. Running on CPU will be extremely slow!")
print("=" * 60)

## Cell 2: Fast Path Resolution & File Existence Checks (< 0.05s)
Instantly resolves exact dataset paths without slow recursive disk crawling.

In [ ]:
# Fast direct lookup function (O(1) existence checks, no slow glob walking)
def resolve_fast(candidates, is_dir=False):
    for path in candidates:
        if os.path.exists(path):
            if is_dir and os.path.isdir(path):
                return path
            elif not is_dir and os.path.isfile(path):
                return path
    for path in candidates:
        if os.path.exists(path):
            return path
    return candidates[0]  # Fallback to first path

STEP2_PATH = resolve_fast([
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/result_step_2.json",
    "/kaggle/input/agentic-vqa-ds/result_step_2.json",
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/result_step2.json",
    "/kaggle/input/agentic-vqa-ds/result_step2.json",
    "pre helper/res/result_step2.json",
])

STEP3_PATH = resolve_fast([
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/step3_predictions.json",
    "/kaggle/input/agentic-vqa-ds/step3_predictions.json",
    "/kaggle/working/step3_results/step3_predictions.json",
    "pre helper/res/step3_predictions.json",
])

SAMPLE_PATH = resolve_fast([
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_test_sample_1000.csv",
    "/kaggle/input/agentic-vqa-ds/infoseek_test_sample_1000.csv",
    "pre helper/files/infoseek_test_sample_1000.csv",
])

KB_PATH = resolve_fast([
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_100k_wiki/wiki_100_dict_v4.json",
    "/kaggle/input/agentic-vqa-ds/infoseek_100k_wiki/wiki_100_dict_v4.json",
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/wiki_100_dict_v4.json",
    "/kaggle/input/agentic-vqa-ds/wiki_100_dict_v4.json",
    "pre helper/files/wiki_100_dict_v4.json",
])

QUERY_IMG_DIR = resolve_fast([
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_val_1000_images/infoseek_val_1000_images",
    "/kaggle/input/agentic-vqa-ds/infoseek_val_1000_images/infoseek_val_1000_images",
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_val_1000_images",
    "/kaggle/input/agentic-vqa-ds/infoseek_val_1000_images",
    "pre helper/files/infoseek_val_1000_images",
], is_dir=True)

OUTPUT_DIR = "/kaggle/working/agentic_vqa_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("SANITY CHECK 2: FILE PATH RESOLUTION")
print("=" * 60)
for label, path in [("Step 2 Results", STEP2_PATH), ("Step 3 Predictions", STEP3_PATH), ("Sample Metadata", SAMPLE_PATH), ("Knowledge Base", KB_PATH), ("Query Images Dir", QUERY_IMG_DIR)]:
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    sz = f"({os.path.getsize(path)/(1024*1024):.1f} MB)" if (exists and os.path.isfile(path)) else ""
    print(f"  {label:<18}: [{status}] {path} {sz}")
print(f"Output Directory  : {OUTPUT_DIR}")
print("=" * 60)

## Cell 3: Load Data, Exact OMGM Metric, and Baseline Margin Analysis
Defines the exact official benchmark evaluation metric (`check_match`) with string normalization, alias matching, and numerical tolerance.

In [ ]:
with open(STEP2_PATH, "r", encoding="utf-8") as f:
    step2_results = json.load(f)
with open(STEP3_PATH, "r", encoding="utf-8") as f:
    step3_baseline = json.load(f)

sample_meta = {}
if os.path.exists(SAMPLE_PATH):
    if SAMPLE_PATH.endswith(".csv"):
        df_sample = pd.read_csv(SAMPLE_PATH)
        for _, row in df_sample.iterrows():
            did = str(row.get("data_id", "")).strip()
            sample_meta[did] = {
                "question": str(row.get("question", "")),
                "image_id": str(row.get("dataset_image_ids", "") or row.get("image_id", "")).split("|")[0].strip(),
                "ground_truth_answers": str(row.get("answer", "") or row.get("ground_truth_answers", "")),
            }
    else:
        with open(SAMPLE_PATH, "r", encoding="utf-8") as f:
            sample_list = json.load(f)
            sample_meta = {str(item.get("data_id")): item for item in sample_list}

# Exact OMGM InfoSeek Evaluation Metric
def normalize_answer(text):
    if not text:
        return ""
    text = str(text).lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = "".join(ch for ch in text if ch not in set(string.punctuation))
    return " ".join(text.split())

def extract_number(text):
    if not text:
        return None
    match = re.search(r'[-+]?\d*\.?\d+', str(text).replace(',', ''))
    if match:
        try:
            return float(match.group())
        except ValueError:
            return None
    return None

def metric_numerical_range(pred_num, ans_num, tolerance=0.10):
    min_val = ans_num * (1.0 - tolerance) if ans_num >= 0 else ans_num * (1.0 + tolerance)
    max_val = ans_num * (1.0 + tolerance) if ans_num >= 0 else ans_num * (1.0 - tolerance)
    low, high = min(min_val, max_val), max(min_val, max_val)
    return low <= pred_num <= high

def check_match(pred, gt_str):
    if not pred or not gt_str:
        return False
    norm_pred = normalize_answer(pred)
    if not norm_pred:
        return False

    if isinstance(gt_str, list):
        raw_list = gt_str
    else:
        raw_list = str(gt_str).split("|")
    gt_answers = [g.strip() for g in raw_list if str(g).strip()]

    # 1. Text & substring match across all aliases
    for gt in gt_answers:
        norm_gt = normalize_answer(gt)
        if norm_gt and (norm_gt == norm_pred or f" {norm_gt} " in f" {norm_pred} " or norm_gt in norm_pred):
            return True

    # 2. Numerical tolerance range (+/- 10%) per OMGM benchmark
    p_num = extract_number(pred)
    clean_cand = str(pred).strip()
    is_pure_digit = bool(re.fullmatch(r"[-+]?\d*\.?\d+", clean_cand))
    if p_num is not None and is_pure_digit:
        for raw_cand in gt_answers:
            c_num = extract_number(raw_cand)
            if c_num is not None and metric_numerical_range(p_num, c_num, tolerance=0.10):
                return True
    return False

# Analyze Baseline Margins & Accuracy
deltas = []
correct_baseline = 0
for did, p in step3_baseline.items():
    s2 = step2_results.get(did, {})
    sc = s2.get("retrieval_similarities", [])
    delta = float(sc[0] - sc[1]) if len(sc) > 1 else 0.0
    deltas.append(delta)
    gt_val = p.get("ground_truth_answers") or sample_meta.get(did, {}).get("ground_truth_answers", "")
    if check_match(p.get("prediction", ""), gt_val):
        correct_baseline += 1

total_q = len(step3_baseline)
print("=" * 60)
print("SANITY CHECK 3: BASELINE MARGIN STATISTICS")
print("=" * 60)
print(f"Total Queries Evaluated : {total_q}")
print(f"Baseline VQA Accuracy   : {correct_baseline/total_q*100:.2f}% ({correct_baseline}/{total_q})")
print(f"Mean Score Margin Delta : {np.mean(deltas):.4f} (Median: {np.median(deltas):.4f})")
print("-" * 60)
print("{:<15} | {:<12} | {:<12} | {:<20}".format("Threshold tau", "Triggered", "Trigger Rate", "Fast Path (0 cost)"))
print("-" * 60)
for t_val in [0.010, 0.015, 0.020, 0.025, 0.030]:
    trig = sum(1 for d in deltas if d < t_val)
    print("tau = {:<10.3f} | {:4d} / {:4d} | {:5.1f}%       | {:4d} queries ({:.1f}%)".format(
        t_val, trig, total_q, trig/total_q*100, total_q - trig, 100 - trig/total_q*100
    ))
print("=" * 60)

## Cell 4: Knowledge Base & Neural BGE Section Reranker
Loads Wikipedia articles and initializes BGE Section Reranker (`BAAI/bge-reranker-v2-m3` in FP16, VRAM ~560 MB) to select the true evidence section for BOTH Candidate 1 (fixing Category C) and Candidate 2 (fixing Category B).

In [ ]:
class KnowledgeEntry:
    def __init__(self, title, url, section_titles, section_texts):
        self.title = title
        self.url = url
        self.section_titles = section_titles
        self.section_texts = section_texts

def reconstruct_wiki_sections(entry):
    sections = []
    if not entry:
        return sections
    for s_title, s_text in zip(entry.section_titles, entry.section_texts):
        sec_lower = s_title.lower()
        if "external links" in sec_lower or "references" in sec_lower:
            continue
        formatted = f"# Wiki Article: {entry.title}\n## Section Title: {s_title}\n{s_text}"
        sections.append(formatted)
    return sections

print(f"Loading Knowledge Base from {KB_PATH}...")
t0 = time.time()
with open(KB_PATH, "r", encoding="utf-8") as f:
    raw_kb = json.load(f)

kb_dict = {
    url: KnowledgeEntry(
        title=info.get("title", ""),
        url=url,
        section_titles=info.get("section_titles", []),
        section_texts=info.get("section_texts", []),
    )
    for url, info in raw_kb.items()
}
del raw_kb
gc.collect()
print(f"Loaded Knowledge Base with {len(kb_dict):,} articles in {time.time()-t0:.1f}s.")

# Load BGE Cross-Encoder for Accurate Section Selection
from transformers import AutoTokenizer, AutoModelForSequenceClassification
bge_model = None
bge_tokenizer = None
try:
    bge_id = "BAAI/bge-reranker-v2-m3"
    print(f"Loading BGE Section Reranker ({bge_id})...")
    bge_tokenizer = AutoTokenizer.from_pretrained(bge_id)
    bge_model = AutoModelForSequenceClassification.from_pretrained(bge_id).half().to("cuda").eval()
    print("BGE Section Reranker loaded successfully in FP16 (VRAM ~560 MB)!")
except Exception as e:
    print(f"Notice: BGE Reranker not loaded ({e}), using lexical matching fallback.")

def select_best_section(question, entry):
    """Accurately selects the Wikipedia section answering the question (never blindly takes index 0)."""
    if not entry:
        return "", -999.0
    sections = reconstruct_wiki_sections(entry)
    if not sections:
        return "", -999.0
    if len(sections) == 1:
        return sections[0], 0.0
    
    # 1. Neural Cross-Encoder (BGE)
    if bge_model is not None and torch.cuda.is_available():
        try:
            pairs = [[question, s[:1500]] for s in sections]
            inputs = bge_tokenizer(pairs, padding=True, truncation=True, return_tensors="pt", max_length=512).to("cuda")
            with torch.no_grad():
                scores = bge_model(**inputs).logits.view(-1).float().cpu().tolist()
            best_idx = int(np.argmax(scores))
            return sections[best_idx], float(scores[best_idx])
        except Exception:
            pass
    
    # 2. Lexical Keyword Fallback
    stopwords = {'what', 'is', 'the', 'of', 'this', 'in', 'how', 'many', 'a', 'an', 'to', 'for', 'does', 'do', 'which', 'where', 'who', 'when', 'why'}
    q_words = [w.lower() for w in re.findall(r'\b\w+\b', question) if w.lower() not in stopwords]
    best_sec = sections[0]
    best_score = -1.0
    for sec in sections:
        sec_lower = sec.lower()
        score = 0.0
        lines = sec_lower.split('\n')
        title_line = lines[1] if len(lines) > 1 else ''
        for w in q_words:
            if w in title_line:
                score += 5.0
            if w in sec_lower:
                score += 1.0
        if score > best_score:
            best_score = score
            best_sec = sec
    return best_sec, best_score

## Cell 5: Load LLaVA-1.5-7B in 4-bit (VRAM Footprint ~4.5 GB)
Uses `BitsAndBytesConfig` for memory-efficient execution on Kaggle T4 GPU.

In [ ]:
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

print("=" * 60)
print("SANITY CHECK 5: INITIALIZING LLAVA-1.5-7B IN 4-BIT")
print("=" * 60)
model_id = "llava-hf/llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

generator_model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
).eval()

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated(0) / (1024**3)
    res = torch.cuda.memory_reserved(0) / (1024**3)
    print(f"LLaVA-1.5-7B Loaded Successfully!")
    print(f"VRAM Allocated : {alloc:.2f} GB (Reserved: {res:.2f} GB)")
    print(f"Available VRAM  : {torch.cuda.get_device_properties(0).total_memory/(1024**3) - res:.2f} GB")
print("=" * 60)

## Cell 6: De-biased Visual Choice & Unit-Aware Generation Helpers + Pilot Test
Implements:
1. `ask_visual_choice_debiased`: Two-pass symmetric routing to neutralize Option-(A) primacy bias without test-time leakage.
2. `generate_answer`: Unit-aware OMGM prompt formatting short factual answers.

In [ ]:
TAU = 0.025  # Empirical ambiguity threshold

def get_query_image(image_id, search_dir):
    clean_id = str(image_id).split("|")[0].strip()
    cands = [
        os.path.join(search_dir, clean_id),
        os.path.join(search_dir, f"{clean_id}.jpg"),
        os.path.join(search_dir, f"{clean_id}.png"),
        os.path.join(search_dir, f"oven_{clean_id}.jpg"),
    ]
    for p in cands:
        if os.path.isfile(p):
            try:
                return PIL.Image.open(p).convert("RGB")
            except Exception:
                pass
    return PIL.Image.new("RGB", (224, 224), color=(128, 128, 128))

def ask_visual_choice_debiased(raw_img, title1, title2):
    """Two-pass symmetric evaluation to neutralize Option-(A) primacy bias."""
    # Pass 1: A=title1, B=title2
    p1 = (
        f"Look at the image carefully.\n"
        f"Which entity does this image depict?\n"
        f"Option (A): {title1}\n"
        f"Option (B): {title2}\n"
        f"Answer strictly with (A) or (B):"
    )
    prompt1 = f"USER: <image>\n{p1}\nASSISTANT:"
    inp1 = processor(images=raw_img, text=prompt1, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out1 = generator_model.generate(**inp1, max_new_tokens=5, do_sample=False)
    dec1 = processor.decode(out1[0][inp1.input_ids.shape[1]:], skip_special_tokens=True).strip().lower()

    # Pass 2: A=title2, B=title1
    p2 = (
        f"Look at the image carefully.\n"
        f"Which entity does this image depict?\n"
        f"Option (A): {title2}\n"
        f"Option (B): {title1}\n"
        f"Answer strictly with (A) or (B):"
    )
    prompt2 = f"USER: <image>\n{p2}\nASSISTANT:"
    inp2 = processor(images=raw_img, text=prompt2, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out2 = generator_model.generate(**inp2, max_new_tokens=5, do_sample=False)
    dec2 = processor.decode(out2[0][inp2.input_ids.shape[1]:], skip_special_tokens=True).strip().lower()

    def parse_choice(dec_str, opt_a, opt_b):
        if "(a)" in dec_str or dec_str.startswith("a") or dec_str == "a":
            return "A"
        if "(b)" in dec_str or dec_str.startswith("b") or dec_str == "b":
            return "B"
        a_low, b_low = opt_a.lower(), opt_b.lower()
        if a_low in dec_str and b_low not in dec_str:
            return "A"
        if b_low in dec_str and a_low not in dec_str:
            return "B"
        return None

    c1 = parse_choice(dec1, title1, title2)
    c2 = parse_choice(dec2, title2, title1)

    p1_chose_cand2 = (c1 == "B")
    p2_chose_cand2 = (c2 == "A")
    p2_chose_cand1 = (c2 == "B")

    # Unanimous vote for Candidate 2 across both permutations
    if p1_chose_cand2 and p2_chose_cand2:
        return "B", f"Unanimous Cand 2 (P1:{dec1} | P2:{dec2})"

    # Pass 1 overcame Option (A) bias to choose Candidate 2 (B), and Pass 2 did not explicitly vote for Candidate 1 (B)
    if p1_chose_cand2 and not p2_chose_cand1:
        return "B", f"Strong Cand 2 (P1:{dec1} | P2:{dec2})"

    # Otherwise (including positional tie where both choose A), retain Candidate 1
    return "A", f"P1:{dec1} | P2:{dec2}"

OMGM_INSTRUCTION = (
    "Answer the encyclopedic question about the given image.\n"
    "Don't mention the visual content of image in your output. Directly output the answer of the question according to the Context.\n"
    "If you need to answer questions about numbers or time, please output the corresponding numerical format directly.\n"
    "Pay special attention to units: if the question asks for metres and context mentions centimetres, convert to metres (e.g., 125 cm is 1.25 metres).\n"
    "If the context does not contain the information required to answer the question, you should answer the question using internal model knowledge.\n"
    "There is an example:\n- Context: # Wiki Article: Dolomites\n## Section Title: Dolomites\nThe Dolomites, also known as the Dolomite Mountains, Dolomite Alps or Dolomitic Alps, are a mountain range located in northeastern Italy. The Dolomites are located in the regions of Veneto, Trentino-Alto Adige/Südtirol and Friuli Venezia Giulia, covering an area shared between the provinces of Belluno, Vicenza, Verona, Trentino, South Tyrol, Udine and Pordenone.\n"
    "- Question: Which city or region does this mountain locate in?\n"
    "Just answer the questions , no explanations needed. Short answer is: Province of Belluno\n\n"
)

def generate_answer(raw_img, question, section_text, max_chars=2500):
    """Generates factual answer using OMGM standard prompt from selected section."""
    trimmed = section_text[:max_chars] if len(section_text) > max_chars else section_text
    prompt = OMGM_INSTRUCTION + f"- Context: {trimmed}\n- Question: {question}\nJust answer the questions , no explanations needed. Short answer is:"
    full_prompt = f"USER: <image>\n{prompt}\nASSISTANT:"
    inputs = processor(images=raw_img, text=full_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = generator_model.generate(**inputs, max_new_tokens=32, do_sample=False)
    prompt_len = inputs.input_ids.shape[1]
    return processor.decode(out[0][prompt_len:], skip_special_tokens=True).strip()

# Collect ambiguous queries
ambiguous_qids = []
for did, p in step3_baseline.items():
    s2 = step2_results.get(did, {})
    sc = s2.get("retrieval_similarities", [])
    if len(sc) > 1 and float(sc[0] - sc[1]) < TAU:
        ambiguous_qids.append(did)

print(f"Found {len(ambiguous_qids)} ambiguous queries where Delta < {TAU:.3f}.")
print("=" * 60)
print("PILOT TEST ON FIRST 2 AMBIGUOUS QUERIES:")
print("=" * 60)
for pilot_qid in ambiguous_qids[:2]:
    s2 = step2_results[pilot_qid]
    c1 = s2["retrieved_entities"][0]["title"]
    c2 = s2["retrieved_entities"][1]["title"]
    m = s2["retrieval_similarities"][0] - s2["retrieval_similarities"][1]
    img_id = sample_meta.get(pilot_qid, {}).get("image_id") or pilot_qid
    raw_img = get_query_image(img_id, QUERY_IMG_DIR)
    choice, dec = ask_visual_choice_debiased(raw_img, c1, c2)
    print(f"Query ID       : {pilot_qid}")
    print(f"Question       : {step3_baseline[pilot_qid]['question']}")
    print(f"Option (A)     : {c1}")
    print(f"Option (B)     : {c2}")
    print(f"Visual Choice  : ({choice}) -> {dec}")
    print(f"Old Pred       : {step3_baseline[pilot_qid]['prediction']}")
    print(f"GT Answer      : {step3_baseline[pilot_qid].get('ground_truth_answers', sample_meta.get(pilot_qid, {}).get('ground_truth_answers'))}")
    print("-" * 60)

## Cell 7: Full Agentic Execution Loop with Zero Ground-Truth Access
Strictly unsupervised execution: the controller does NOT access `ground_truth_answers` or `check_match` during inference.

In [ ]:
agentic_predictions = dict(step3_baseline)

for qid, item in agentic_predictions.items():
    item["agent_triggered"] = False
    item["agent_switched_to_b"] = False
    item["action_note"] = "Fast-Path Baseline (Confident)"

switched_to_b_count = 0
cat_c_refreshed_count = 0

print(f"Starting Zero-Leakage Agentic Controller across {len(ambiguous_qids)} ambiguous queries...")
start_time = time.time()

for i, qid in enumerate(tqdm(ambiguous_qids, desc="Agentic Controller")):
    b_info = step3_baseline[qid]
    s2 = step2_results[qid]
    q_text = b_info["question"]

    cands = s2.get("retrieved_entities", [])
    if len(cands) < 2:
        continue

    t1 = cands[0].get("title", "Candidate 1")
    u1 = cands[0].get("url", "")
    t2 = cands[1].get("title", "Candidate 2")
    u2 = cands[1].get("url", "")

    img_id = sample_meta.get(qid, {}).get("image_id") or b_info.get("dataset_image_ids", qid)
    raw_img = get_query_image(img_id, QUERY_IMG_DIR)

    # 1. Retrieve true sections for both Candidate 1 and Candidate 2 using BGE cross-encoder
    sec1_text, score1 = select_best_section(q_text, kb_dict.get(u1))
    if not sec1_text:
        sec1_text = b_info.get("top1_section", "")
        
    sec2_text, score2 = select_best_section(q_text, kb_dict.get(u2))
    if not sec2_text:
        sec2_text = reconstruct_wiki_sections(kb_dict.get(u2))[0] if (kb_dict.get(u2) and len(reconstruct_wiki_sections(kb_dict.get(u2))) > 0) else ""

    # 2. De-biased Visual Choice (neutralizing Option-(A) primacy bias)
    choice, dec = ask_visual_choice_debiased(raw_img, t1, t2)

    # Decision logic (strictly unsupervised):
    # Switch to Candidate 2 if visually confirmed OR if Candidate 2's cross-encoder section score is overwhelmingly higher
    switch_to_b = (choice == "B") or (score2 > score1 + 2.0 and score2 > 0.0)

    new_pred = b_info["prediction"]
    action_note = "Kept Candidate 1 (Baseline Evidence)"

    if switch_to_b:
        switched_to_b_count += 1
        action_note = f"Switched to {t2} (Cat B Branch: {dec})"
        new_pred = generate_answer(raw_img, q_text, sec2_text)
    else:
        # Category C: If keeping Candidate 1, check if BGE identified a superior section
        # (Unsupervised: section differs from baseline and cross-encoder score is high)
        if sec1_text and sec1_text != b_info.get("top1_section", "") and score1 > 1.0:
            cat_c_refreshed_count += 1
            action_note = f"Refreshed Section for {t1} (Cat C BGE Score: {score1:.2f})"
            new_pred = generate_answer(raw_img, q_text, sec1_text)

    # Record Agent Prediction Entry (No Oracle Rollback!)
    updated = dict(b_info)
    updated["prediction"] = new_pred
    updated["agent_triggered"] = True
    updated["agent_switched_to_b"] = switch_to_b
    updated["agent_score_margin"] = float(s2["retrieval_similarities"][0] - s2["retrieval_similarities"][1])
    updated["agent_options"] = [t1, t2]
    updated["action_note"] = action_note
    agentic_predictions[qid] = updated

elapsed = time.time() - start_time
print(f"\nCompleted Agentic Recovery in {elapsed:.1f}s ({elapsed/len(ambiguous_qids):.2f}s per query).")
print(f"Switched to Candidate 2: {switched_to_b_count} times.")
print(f"Category C Section Refreshes: {cat_c_refreshed_count} times.")

## Cell 8: Final Zero-Leakage Comparative Evaluation Table & Results Export
Evaluates both baseline and agentic predictions against ground-truth only at the very end, computing genuine recoveries, degradations, and net gain.

In [ ]:
baseline_correct_count = 0
final_correct = 0
recovered_events = []
degraded_events = []
preserved_correct = 0

for qid in step3_baseline.keys():
    b_info = step3_baseline[qid]
    a_info = agentic_predictions[qid]
    gt_ans = a_info.get("ground_truth_answers") or sample_meta.get(qid, {}).get("ground_truth_answers", "")

    was_correct = check_match(b_info.get("prediction", ""), gt_ans)
    is_now_correct = check_match(a_info.get("prediction", ""), gt_ans)

    a_info["is_correct"] = is_now_correct
    if was_correct:
        baseline_correct_count += 1
    if is_now_correct:
        final_correct += 1

    if not was_correct and is_now_correct:
        recovered_events.append({
            "qid": qid,
            "question": b_info.get("question", ""),
            "action": a_info.get("action_note", ""),
            "old_pred": b_info.get("prediction", ""),
            "new_pred": a_info.get("prediction", ""),
            "gt": gt_ans,
        })
    elif was_correct and not is_now_correct:
        degraded_events.append({
            "qid": qid,
            "question": b_info.get("question", ""),
            "action": a_info.get("action_note", ""),
            "old_pred": b_info.get("prediction", ""),
            "new_pred": a_info.get("prediction", ""),
            "gt": gt_ans,
        })
    elif was_correct and is_now_correct:
        preserved_correct += 1

final_acc = (final_correct / total_q) * 100.0
baseline_acc = (baseline_correct_count / total_q) * 100.0
delta_acc = final_acc - baseline_acc
net_recovered = final_correct - baseline_correct_count

avg_tool_calls = 1.0 + (len(ambiguous_qids) / total_q)

print("=" * 75)
print("FINAL AGENTIC RETRIEVAL CONTROLLER EVALUATION RESULTS (ZERO-LEAKAGE)")
print("=" * 75)
print(f"{'Metric':<32} | {'Baseline (OMGM)':<16} | {'Ours (Agentic)':<16} | {'Net Change'}")
print("-" * 75)
print(f"{'Overall VQA Accuracy':<32} | {baseline_acc:6.2f}% ({baseline_correct_count}/{total_q}) | {final_acc:6.2f}% ({final_correct}/{total_q}) | {delta_acc:+6.2f}% ({net_recovered:+d} queries)")
print(f"{'Triggered Queries (Delta < tau)':<32} | {'0 / 1000 (0.0%)':<16} | {len(ambiguous_qids):4d} / {total_q} ({len(ambiguous_qids)/total_q*100:4.1f}%) | Ambiguity Isolated")
print(f"{'Gross Recovered Queries':<32} | {'0 (Rigid Fixed)':<16} | {len(recovered_events):4d} Recovered      | Wrong -> Right")
print(f"{'Degraded Queries':<32} | {'0 (Rigid Fixed)':<16} | {len(degraded_events):4d} Degraded       | Right -> Wrong")
print(f"{'Net Error Improvement':<32} | {'0 (Rigid Fixed)':<16} | {net_recovered:+4d} Net Gain       | Honest Net Effect")
print(f"{'Average Generator Calls / Query':<32} | {'1.00 calls':<16} | {avg_tool_calls:4.2f} calls       | Minimal +{avg_tool_calls-1.0:.2f} overhead")
print("=" * 75)

with open(os.path.join(OUTPUT_DIR, "agentic_predictions.json"), "w", encoding="utf-8") as f:
    json.dump(agentic_predictions, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "agentic_recovered_queries.json"), "w", encoding="utf-8") as f:
    json.dump({
        "recovered_events": recovered_events,
        "degraded_events": degraded_events,
    }, f, indent=2)

summary = {
    "baseline_accuracy": round(baseline_acc, 2),
    "agentic_accuracy": round(final_acc, 2),
    "accuracy_gain": round(delta_acc, 2),
    "total_queries": total_q,
    "baseline_correct_count": baseline_correct_count,
    "agentic_correct_count": final_correct,
    "net_recovered_queries": net_recovered,
    "gross_recovered_queries": len(recovered_events),
    "degraded_queries": len(degraded_events),
    "preserved_correct_queries": preserved_correct,
    "ambiguous_queries": len(ambiguous_qids),
    "trigger_rate": round(len(ambiguous_qids) / total_q * 100.0, 2),
    "switched_to_b_count": switched_to_b_count,
    "cat_c_refreshed_count": cat_c_refreshed_count,
    "average_calls_per_query": round(avg_tool_calls, 2),
}
with open(os.path.join(OUTPUT_DIR, "agentic_results_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"All artifacts saved to {OUTPUT_DIR}/")